# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [1]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --

# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings, load_items
# -- add new imports here --
from surprise import accuracy
from surprise.model_selection import cross_validate, train_test_split, LeaveOneOut
from models import get_top_n  
import random as rd

# 1. Try the loader with surprise_format set to True
data = load_ratings(surprise_format=True)

# 2. Verify the results
print(f"Object Type: {type(data)}")

# 3. Build a trainset to confirm data is correctly loaded (check)
trainset = data.build_full_trainset()
print(f"Number of ratings: {trainset.n_ratings}")
print(f"Number of users: {trainset.n_users}")
print(f"Number of items: {trainset.n_items}")

print("Loader successfully tested in Surprise format!")

Object Type: <class 'surprise.dataset.DatasetAutoFolds'>
Number of ratings: 381181
Number of users: 1000
Number of items: 8737
Loader successfully tested in Surprise format!


# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [2]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    """Generate predictions on a random test set specified in eval_config"""
    trainset, testset = train_test_split(
        ratings_dataset, 
        test_size=eval_config.test_size, 
        random_state=42
    )
    algo.fit(trainset)
    predictions = algo.test(testset)
    return predictions

def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    loo = LeaveOneOut(n_splits=1, random_state=1)
    for trainset, testset in loo.split(ratings_dataset):
        algo.fit(trainset)
        anti_testset = trainset.build_anti_testset()
        predictions = algo.test(anti_testset)
        anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user with full training set (LOO)"""
    full_trainset = ratings_dataset.build_full_trainset()
    algo.fit(full_trainset)
    anti_testset = full_trainset.build_anti_testset()
    predictions = algo.test(anti_testset)
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n


#########################################
# Test of the three functions as required
#########################################
from models import ModelBaseline1

eval_config = EvalConfig()
data = load_ratings(surprise_format=True)
algo = ModelBaseline1()

predictions_a = generate_split_predictions(algo, data, eval_config)
print(f"Test (a): Success. {len(predictions_a)} raw predictions generated.")

top_n_b, testset_b = generate_loo_top_n(algo, data, eval_config)
print(f"Test (b): Success. {len(top_n_b)} users with recommendations.")
for uid, recs in list(top_n_b.items())[:3]: 
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")

top_n_c = generate_full_top_n(algo, data, eval_config)
print(f"Test (c): Success. {len(top_n_c)} users with recommendations.")
for uid, recs in list(top_n_c.items())[:3]:  
    print(f"  User {uid}: {len(recs)} recommendations -> {recs}")


def precompute_information(df_ratings, df_items):
    """ Returns a dictionary that precomputes relevant information for evaluating in full mode
    
    Dictionary keys:
    - precomputed_dict["item_to_rank"]   : dict mapping movie_id -> popularity rank
    - precomputed_dict["item_freq"]      : dict mapping movie_id -> number of users who rated it
    - precomputed_dict["n_users"]        : total number of unique users
    - precomputed_dict["genre_vectors"]  : dict mapping movie_id -> binary genre vector (np.array)
    """
    precomputed_dict = {}
    
    # item_to_rank: rank 1 = most frequently rated movie
    movie_counts = df_ratings[C.ITEM_ID_COL].value_counts()
    item_to_rank = movie_counts.rank(ascending=False, method='first').to_dict()
    precomputed_dict["item_to_rank"] = item_to_rank

    # item_freq: number of distinct users who rated each item
    item_freq = df_ratings.groupby(C.ITEM_ID_COL)[C.USER_ID_COL].nunique().to_dict()
    precomputed_dict["item_freq"] = item_freq

    # n_users: total distinct users in the dataset
    precomputed_dict["n_users"] = df_ratings[C.USER_ID_COL].nunique()

    # genre_vectors: binary genre vector per item (built from df_items)
    precomputed_dict["genre_vectors"] = build_genre_vectors(df_items)

    return precomputed_dict                


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

Test (a): Success. 95296 raw predictions generated.
Test (b): Success. 1000 users with recommendations.
  User 277: 40 recommendations -> [(2791, 2), (4284, 2), (4420, 2), (4091, 2), (5040, 2), (60832, 2), (92694, 2), (3805, 2), (72, 2), (127202, 2), (83, 2), (96911, 2), (4130, 2), (468, 2), (5682, 2), (176, 2), (7228, 2), (2282, 2), (3197, 2), (2022, 2), (43921, 2), (1532, 2), (4932, 2), (48516, 2), (32289, 2), (8511, 2), (2581, 2), (4621, 2), (32139, 2), (4713, 2), (3543, 2), (8133, 2), (25828, 2), (6663, 2), (94, 2), (1904, 2), (41527, 2), (690, 2), (4186, 2), (4964, 2)]
  User 605: 40 recommendations -> [(3556, 2), (6423, 2), (7130, 2), (2635, 2), (1717, 2), (2528, 2), (73808, 2), (3605, 2), (63113, 2), (4503, 2), (4898, 2), (141956, 2), (2864, 2), (7084, 2), (2315, 2), (2866, 2), (64508, 2), (2688, 2), (111384, 2), (1801, 2), (1911, 2), (8821, 2), (5853, 2), (3217, 2), (6568, 2), (6477, 2), (72405, 2), (1920, 2), (71520, 2), (79868, 2), (2301, 2), (25769, 2), (5644, 2), (96432, 2)

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [3]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric)
    
    A hit (1) happens when the movie in the testset has been picked by the top-n recommender
    A fail (0) happens when the movie in the testset has not been picked by the top-n recommender
    """
    hits = 0
    total_users = len(testset)

    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recommendations = [item_id for (item_id, _) in anti_testset_top_n[user_id]]
            if movie_id in recommendations:
                hits += 1

    hit_rate = hits / total_users if total_users > 0 else 0
    return hit_rate


def get_ndcg(anti_testset_top_n, testset):
    """NDCG@K: Normalized Discounted Cumulative Gain (LOO metric).

    In the LOO setting, each user has exactly one hidden item (binary relevance).
    If the hidden item appears at rank k in the top-K list:
        NDCG_u = 1 / log2(k + 1)
    If not found in top-K:
        NDCG_u = 0
    IDCG = 1  (ideal: hidden item at rank 1 → 1/log2(2) = 1)

    References:
        Cremonesi et al. (2010) — Performance of Recommender Algorithms on Top-N Tasks
        He et al. (2017) — Neural Collaborative Filtering
    """
    total_ndcg = 0.0
    total_users = len(testset)

    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recommendations = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recommendations:
                rank = recommendations.index(movie_id) + 1  # 1-indexed
                total_ndcg += 1.0 / np.log2(rank + 1)

    return total_ndcg / total_users if total_users > 0 else 0.0


def get_novelty(anti_testset_top_n, item_to_rank, **kwargs):
    """Compute the average novelty of the top-n recommendation over the users (full metric)
    
    The novelty is defined as the average ranking of the movies recommended
    """
    total_novelty = 0
    total_users = len(anti_testset_top_n)

    for user_id, recommendations in anti_testset_top_n.items():
        user_novelty_sum = 0
        n_items = len(recommendations)
        
        for movie_id, _ in recommendations:
            rank = item_to_rank.get(movie_id, len(item_to_rank))
            user_novelty_sum += rank
        
        if n_items > 0:
            total_novelty += user_novelty_sum

    average_rank_sum = total_novelty / total_users if total_users > 0 else 0
    return average_rank_sum


def get_novelty_miuf(anti_testset_top_n, item_freq, n_users, **kwargs):
    """MIUF: Mean Inverse User Frequency — novelty via inverse popularity.

    MIUF = (1/|R|) * sum_{i in R} [ -log2(|U_i| / |U|) ]
    where |U_i| = number of users who rated item i, |U| = total users.
    Higher MIUF = more niche (novel) recommendations.
    """
    total = 0.0
    count = 0
    for user_id, recommendations in anti_testset_top_n.items():
        for movie_id, _ in recommendations:
            freq = item_freq.get(movie_id, 1)
            total += -np.log2(freq / n_users)
            count += 1
    return total / count if count > 0 else 0.0


def build_genre_vectors(df_items):
    """Build a binary genre vector for each item from the genres column."""
    all_genres = set()
    for genres_str in df_items[C.GENRES_COL].dropna():
        for g in genres_str.split('|'):
            all_genres.add(g)
    all_genres = sorted(all_genres)
    genre_to_idx = {g: i for i, g in enumerate(all_genres)}

    genre_vectors = {}
    for item_id, row in df_items.iterrows():
        vec = np.zeros(len(all_genres))
        if pd.notna(row[C.GENRES_COL]):
            for g in row[C.GENRES_COL].split('|'):
                if g in genre_to_idx:
                    vec[genre_to_idx[g]] = 1.0
        genre_vectors[item_id] = vec
    return genre_vectors


def get_diversity_ild(anti_testset_top_n, genre_vectors, **kwargs):
    """ILD: Intra-List Diversity — average pairwise cosine dissimilarity on genre vectors.

    ILD = (1/|users|) * sum_u [ mean_{i<j} (1 - cos(gi, gj)) ]
    Higher ILD = more genre-diverse recommendation lists.
    """
    total_diversity = 0.0
    total_users = 0

    for user_id, recommendations in anti_testset_top_n.items():
        items = [movie_id for movie_id, _ in recommendations if movie_id in genre_vectors]
        n = len(items)
        if n < 2:
            continue

        user_div = 0.0
        pairs = 0
        for i in range(n):
            for j in range(i + 1, n):
                vi = genre_vectors[items[i]]
                vj = genre_vectors[items[j]]
                norm_i = np.linalg.norm(vi)
                norm_j = np.linalg.norm(vj)
                if norm_i > 0 and norm_j > 0:
                    cos_sim = np.dot(vi, vj) / (norm_i * norm_j)
                    user_div += 1.0 - cos_sim
                pairs += 1

        if pairs > 0:
            total_diversity += user_div / pairs
        total_users += 1

    return total_diversity / total_users if total_users > 0 else 0.0

## Pitfalls of Using a Sum of Ranks as a Novelty Metric

1. **Sensitivity to N (top-n size)**: If two models recommend different numbers
   of items (e.g. 10 vs 40), the sum of ranks will be mechanically higher for
   the one recommending more items, without being truly more "novel".

2. **Linearity of ranks**: The difference between rank 1 and rank 100 is
   treated the same as between rank 1000 and rank 1100. However, moving from
   the most popular movie to the 100th is a far more radical shift in popularity
   than moving from rank 1000 to 1100. A logarithmic scale would better capture
   this reality.

3. **Quality vs. Novelty trade-off**: A model could achieve a very high novelty
   score by recommending the least popular (and potentially worst) movies on the
   platform that nobody wants to watch. Novelty should always be balanced with
   precision metrics (MAE, RMSE, Hit Rate) to ensure recommendations remain
   relevant.

4. **Dependence on catalogue size**: A rank of 500 in a catalogue of 600 movies
   does not have the same meaning as a rank of 500 in a catalogue of 100,000
   movies. The metric is therefore not comparable across systems with catalogues
   of different sizes.

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [4]:
np.random.seed(1)
rd.seed(1)

AVAILABLE_METRICS = {
    "split": {
        "mae":  (accuracy.mae,  {'verbose': False}),
        "rmse": (accuracy.rmse, {'verbose': False}),
    },
    "loo": {
        "hit_rate": (get_hit_rate, {}),
        "ndcg":     (get_ndcg,     {}),
    },
    "full": {
        "novelty": (get_novelty,      {}),
        "miuf":    (get_novelty_miuf,  {}),
        "ild":     (get_diversity_ild, {}),
    }
}

df_ratings_pd = load_ratings(surprise_format=False)
df_items      = load_items()
sp_ratings    = load_ratings(surprise_format=True)
precomputed_dict = precompute_information(df_ratings_pd, df_items)
evaluation_report = create_evaluation_report(EvalConfig(), sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)
display(evaluation_report)

Handling model baseline_1
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_2
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_3
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model baseline_4
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model UserBased_Manual
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions
Training full predictions
Handling model ContentBased_ridge_cv
[TMDB] Cache not found at data\hackathon\content\tmdb_cache.json. Run 'python fetch_tmdb.py' first.
Training split predictions
- computing metric rmse
- computing metric mae
Training loo predictions


KeyboardInterrupt: 

## Evaluation Report Observations

**Note**: Results obtained on the test dataset (6 users, 10 items),
used only to verify that the pipeline is working correctly.

### 1. Precision Metrics (MAE & RMSE)
- **Baseline 4 (SVD)** is the best performing model on MAE (0.954). This is
  consistent since SVD is a learning algorithm that minimizes prediction error,
  unlike the static baselines.
- **Baseline 1** is the least performant (MAE=1.125): always predicting the same
  value captures none of the nuances of user preferences.
- RMSE is systematically higher than MAE for all models, which is expected:
  RMSE penalizes large errors more heavily (squared differences).

### 2. Hit Rate (1.0)
- All 4 models display a Hit Rate of 1.0 (100%), which would be impossible
  in a real scenario.
- This is explained by the dataset size: with only ~10 available movies and
  a top_n_value=40, the model will always include the "hidden" movie in its list.
- The metric is correctly implemented, but will only be discriminant on a larger
  dataset (thousands of movies, with only 40 possible recommendations).

### 3. Novelty (~30.17)
- The value is nearly identical across all baselines.
- For models predicting constant or average values (Baseline 1 and 3), the
  ordering of recommendations depends on item appearance order or tie-breaking.
- On such a small dataset, all models end up recommending every available movie,
  making the average popularity rank mechanically identical for everyone.

## Content-Based Models Evaluation

### 1. Random Sample vs. Random Score
Random Sample (RMSE 1.31) outperforms Random Score (RMSE 1.79). Random Sample draws from the user's own rating distribution, centered around their personal mean (~3.5 on MovieLens), whereas Random Score samples uniformly in [0.5, 5] (mean ~2.75). Predictions from Random Sample are therefore structurally closer to real ratings.

### 2. Linear Regression (Intercept True vs. False)
Linear Regression with `fit_intercept=True` (RMSE 0.93) vastly outperforms `fit_intercept=False` (RMSE 1.55). Without an intercept, the model must explain a ~3.5-star average using only `coef × title_length`, which is impossible without distorting the coefficient. The intercept captures the user's baseline rating tendency — essentially their mean. With a single weak feature like title length, the intercept does almost all the work: 0.93 is roughly what a "predict the user's mean" baseline would achieve.

### 3. Comparing more advanced models
As base features, we used `all_content_tmdb_tags2000`, which concatenates genome scores (1128 dims), a rich TF-IDF on user tags (up to 2000 features, bigrams, sublinear TF), normalized release year + decade one-hot, genres (TF-IDF), and TMDB metadata (runtime, language, country, studio, directors, cast, keywords, collection, budget, writers, release date, overview TF-IDF). The total feature space spans several thousand dimensions. We compared two regressors:
- **Ridge**: L2-regularized linear regression with a fixed `alpha=1.0` for all users.
- **RidgeCV**: same model, but `alpha` is selected per user via cross-validation over `1e-4` to `1e5`.

**RidgeCV (0.74) outperforms standard Ridge (0.93)** because with several thousand features and only tens to hundreds of ratings per user, `alpha=1.0` is severely under-regularized: Ridge overfits and falls back to roughly the same RMSE as a "predict the user's mean" model. RidgeCV picks larger alphas for sparse profiles (strong shrinkage) and smaller ones for rich profiles, adapting regularization to each user's data volume. Notably, **RidgeCV (0.744) even beats the best collaborative baseline so far, SVD (0.817)**, using only item content and per-user ridge profiles.